# CVPR21_PASS on IP102 Dataset - Kaggle Notebook

This notebook runs the CVPR21_PASS (CVS) method on the IP102 dataset for lifelong learning / class-incremental retrieval.

## Repository
**Default**: `https://github.com/nta2112/CVS-custom-IP102`

## Setup
1. **Add IP102 dataset** as a Kaggle Input (named `ip102-dataset` or similar)
2. **Optional**: Set `IP102_CODE_REPO` environment variable to override the default repo
3. **Run all cells** in order

## Dataset Structure Expected
The input dataset should contain:
- `train.json`, `val.json`, `test.json` (COCO format)
- `filtered_class.txt` (25 class IDs, one per line)
- `classes.txt` (102 class names mapping)
- `VOC2007/VOC2007/JPEGImages/` (image files)

## 1. Clone Repository & Setup Environment

**Default repo**: `https://github.com/nta2112/CVS-custom-IP102`

You can override by setting the `IP102_CODE_REPO` environment variable before running this cell.

In [ ]:
import os
import sys
import subprocess
import json
import glob
import pandas as pd

# Default repo URL (can be overridden by IP102_CODE_REPO env var)
DEFAULT_REPO = 'https://github.com/nta2112/CVS-custom-IP102'
code_repo = os.environ.get('IP102_CODE_REPO', DEFAULT_REPO)

repo_dir = '/kaggle/working/CVPR21_PASS'
if not os.path.exists(repo_dir):
    print('Cloning from {}...'.format(code_repo))
    subprocess.run(['git', 'clone', code_repo, repo_dir], check=True)
else:
    print('Repo exists, pulling latest changes...')
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)

os.chdir(repo_dir)
# Ensure origin points to the correct repo
subprocess.run(['git', 'remote', 'remove', 'origin'], capture_output=True)
subprocess.run(['git', 'remote', 'add', 'origin', code_repo], capture_output=True)

sys.path.insert(0, os.getcwd())
print('Working directory: {}'.format(os.getcwd()))
print('Repo: {}'.format(code_repo))

In [ ]:
# Install dependencies
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'faiss-cpu', 'scikit-learn', 'tqdm', 'pandas', 'matplotlib',
    'easydict', 'randaugment', 'pretrainedmodels'])

In [ ]:
# Check GPU
import torch
print('PyTorch: {}'.format(torch.__version__))
print('CUDA available: {}'.format(torch.cuda.is_available()))
print('GPU count: {}'.format(torch.cuda.device_count()))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print('  GPU {}: {}'.format(i, torch.cuda.get_device_name(i)))

In [ ]:
# Verify dataset location
import os

def find_dataset():
    candidates = [
        '/kaggle/input/ip102-dataset',
        '/kaggle/input/ip102',
        '/kaggle/input/IP102',
        '/kaggle/input/ip102-dataset/IP102 dataset',
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.json')) and os.path.exists(os.path.join(c, 'filtered_class.txt')):
            return c
    # Deep search
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.json' in files and 'filtered_class.txt' in files:
            return root
    return None

dataset_root = find_dataset()
if dataset_root:
    os.environ['IP102_DATA_ROOT'] = dataset_root
    print('Found dataset at: {}'.format(dataset_root))
    print('Files: {}'.format(os.listdir(dataset_root)[:10]))
else:
    print('WARNING: Dataset not found in /kaggle/input. Make sure to add it as an Input.')

In [ ]:
# Quick test: verify imports work
import sys
sys.path.insert(0, os.getcwd())

from loader.ip102 import IP102, find_ip102_root
from utils.train_utils import select_model, unwrap_model, get_loader_kwargs
from metrics.retrieval import RetrievalMetric
from metrics.openworld import OpenWorldMetric
from metrics.lifelong import LifelongMetric, save_results_csv, save_history_json

print('All imports OK')

# Test dataset loading
ds = IP102(mode='train', session_id=0, exp_name='disjoint')
print('Session 0: {} samples, labels: {}'.format(len(ds), sorted(set(ds.targets))))

# Test model
model = select_model('resnet18', 'ip102', num_classes=7, feature_size=128, nsloss=True, pretrain=False)
print('Model: {}'.format(type(model).__name__))

import torch
x = torch.randn(2, 3, 224, 224)
model.eval()
with torch.no_grad():
    feats, logit = model(x)
print('Forward OK: feats={}, logit={}'.format(feats.shape, logit.shape))

In [ ]:
# Run training using subprocess (better than exec for argparse scripts)
def run_session(session_id, prev_dir=None, n_epochs=1, replay=False, buffer=2000, bs=32):
    save_dir = 'exp/ip102_disjoint_cvs{}'.format(session_id)
    
    cmd = [
        sys.executable, 'train.py',
        '--dataset', 'ip102',
        '--exp_name', 'disjoint',
        '--arch', 'resnet18',
        '--embed_dim', '128',
        '--bs', str(bs),
        '--lr', '0.03',
        '--n_epochs', str(n_epochs),
        '--session_id', str(session_id),
        '--exp_dir', 'exp',
        '--save_dir', save_dir,
        '--kernels', '4',
    ]
    
    if session_id > 0 and prev_dir:
        cmd.extend(['--load_dir', prev_dir])
    if replay:
        cmd.extend(['--replay', '--buffer', str(buffer)])
    
    print('\n=== Running session {} ==='.format(session_id))
    print('Command: {}'.format(' '.join(cmd)))
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Session {} failed with code {}'.format(session_id, result.returncode))
    return save_dir

# Run session 0 (quick test with 1 epoch)
save_dir_0 = run_session(0, n_epochs=1)

In [ ]:
# Run remaining sessions (1-3) with replay
prev_dir = save_dir_0
for session_id in range(1, 4):
    prev_dir = run_session(session_id, prev_dir=prev_dir, n_epochs=1, replay=True, buffer=2000, bs=32)

In [ ]:
# Full training function (run all 4 tasks with 256 epochs each)
def run_train(model='CVS', max_tasks=0, memory_size=2000, epochs=256):
    """
    Run training for IP102 dataset.

    Args:
        model: Model name (currently only 'CVS' supported)
        max_tasks: Maximum number of tasks to run (0 = all 4 tasks)
        memory_size: Replay buffer size
        epochs: Number of epochs per task

    Returns:
        Final results DataFrame
    """
    n_tasks = 4 if max_tasks == 0 else min(max_tasks, 4)
    prev_dir = None
    
    for session_id in range(n_tasks):
        save_dir = 'exp/ip102_disjoint_cvs{}'.format(session_id)
        
        cmd = [
            sys.executable, 'train.py',
            '--dataset', 'ip102',
            '--exp_name', 'disjoint',
            '--arch', 'resnet18',
            '--embed_dim', '128',
            '--bs', '32',
            '--lr', '0.03',
            '--n_epochs', str(epochs),
            '--session_id', str(session_id),
            '--exp_dir', 'exp',
            '--save_dir', save_dir,
            '--kernels', '4',
        ]
        
        if session_id > 0 and prev_dir:
            cmd.extend(['--load_dir', prev_dir, '--replay', '--buffer', str(memory_size)])
        
        print('\n=== Running session {} ==='.format(session_id))
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=7200*epochs//256)
        print(result.stdout)
        if result.stderr:
            print('STDERR:', result.stderr)
        if result.returncode != 0:
            raise RuntimeError('Session {} failed with code {}'.format(session_id, result.returncode))
        prev_dir = save_dir
    
    # Load final results
    results_path = 'exp/ip102_disjoint_cvs{}/results.csv'.format(n_tasks-1)
    if os.path.exists(results_path):
        return pd.read_csv(results_path)
    return None

# Example usage (commented out - uncomment to run full training):
# full_results = run_train(model='CVS', max_tasks=0, memory_size=2000, epochs=256)
# print(full_results)

In [ ]:
# Display final results
results_files = glob.glob('exp/ip102_disjoint_cvs*/results.csv')
for rf in sorted(results_files):
    print('\n=== {} ==='.format(rf))
    df = pd.read_csv(rf)
    print(df.to_string(index=False))

In [ ]:
# Display history
import json
history_files = glob.glob('exp/ip102_disjoint_cvs*/history.json')
for hf in sorted(history_files):
    print('\n=== {} ==='.format(hf))
    with open(hf, 'r') as f:
        history = json.load(f)
    print(json.dumps(history, indent=2))

In [ ]:
# Push results to GitHub (optional)
# Uncomment and configure if you want to push results
# import subprocess
# subprocess.run(['git', 'add', 'exp/'])
# subprocess.run(['git', 'commit', '-m', 'Add IP102 results'])
# subprocess.run(['git', 'push', 'origin', 'HEAD'])